# Python FastAPI와 Gemini API로 구현하는 미니 AI 에이전트

ICT폴리텍대학 교육중점교원 시범강의 실습 노트북 (Google Colab)

이 노트북은 설치 없이 브라우저에서 바로 실행합니다. 위에서부터 셀을 차례로 실행하세요.

학습 흐름: 요청 → 검증 → 프롬프트 → 모델 호출 → 응답 반환

- Part 1. 환경 준비와 API 키 등록
- Part 2. Gemini 모델 직접 호출 (개념 이해)
- Part 3. FastAPI로 웹 API 구조 만들기 (서버 측 호출)
- Part 4. 나만의 학습 도우미 만들기 (과제)

## Part 1. 환경 준비

필요한 패키지를 설치합니다. 약 30초 걸립니다.

In [ ]:
!pip install -q google-genai fastapi httpx pydantic

### API 키 등록 (안전한 방법)

API 키는 코드에 직접 적지 않습니다. Colab 왼쪽의 열쇠 모양(보안 비밀) 아이콘을 눌러
이름 `GEMINI_API_KEY`로 키를 등록하고 노트북 액세스를 켭니다.

키 발급: https://aistudio.google.com/apikey

In [ ]:
import os

# Colab Secrets에서 키를 읽어 환경변수로 등록 (코드에 키를 노출하지 않음)
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print('API 키 등록 완료')
except Exception as e:
    print('Colab Secrets에서 키를 찾지 못했습니다. 왼쪽 열쇠 아이콘에서 GEMINI_API_KEY를 등록하세요.')
    print('상세:', e)

In [ ]:
# 사용할 모델 (한 곳에서만 지정 — 모델이 바뀌면 이 줄만 수정)
# 안정 모델명은 https://ai.google.dev/gemini-api/docs/models 에서 확인
GEMINI_MODEL = 'gemini-3-flash'
os.environ['GEMINI_MODEL'] = GEMINI_MODEL
print('사용 모델:', GEMINI_MODEL)

## Part 2. Gemini 모델 직접 호출

먼저 모델을 직접 호출해 응답이 오는지 확인합니다. 가장 단순한 형태입니다.

In [ ]:
from google import genai

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

resp = client.models.generate_content(
    model=GEMINI_MODEL,
    contents='AI 에이전트와 일반 챗봇의 차이를 두 문장으로 설명해줘.',
)
print(resp.text)

### 프롬프트를 설계하면 답변 품질이 달라집니다

AI에게 역할, 답변 길이, 예시 조건을 명확히 주는 함수를 만듭니다.

In [ ]:
def build_prompt(question: str) -> str:
    return f'''너는 ICT폴리텍 학생을 돕는 AI 학습 도우미다.
아래 질문에 대해 초급 학습자도 이해할 수 있게 5문장 이내로 답하라.
가능하면 실무 예시를 1개 포함하라.

질문: {question}'''


def ask_gemini(question: str) -> str:
    resp = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=build_prompt(question),
    )
    return resp.text


print(ask_gemini('스마트 컨트랙트가 무엇인가요?'))

## Part 3. FastAPI로 웹 API 구조 만들기

실무에서는 모델을 직접 부르지 않고 서버를 거칩니다.
요청을 검증하고(`pydantic`), 응답 형식을 고정해 다른 시스템과 연결하기 쉽게 만듭니다.

Colab에서는 포트를 열지 않고 `TestClient`로 서버를 그대로 호출해 구조를 체험합니다.

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

app = FastAPI(title='ICT Mini AI Agent')


class AskRequest(BaseModel):
    question: str = Field(..., min_length=2, max_length=300)


class AskResponse(BaseModel):
    answer: str
    model: str


@app.post('/ask', response_model=AskResponse)
def ask(request: AskRequest) -> AskResponse:
    api_key = os.getenv('GEMINI_API_KEY')
    if not api_key:
        raise HTTPException(status_code=500, detail='GEMINI_API_KEY가 설정되지 않았습니다.')
    answer = ask_gemini(request.question)
    return AskResponse(answer=answer, model=GEMINI_MODEL)


print('FastAPI 앱 정의 완료. 엔드포인트: POST /ask')

In [ ]:
from fastapi.testclient import TestClient

test_client = TestClient(app)

# 정상 요청
r = test_client.post('/ask', json={'question': 'API 키를 코드에 넣으면 안 되는 이유는?'})
print('상태코드:', r.status_code)
print(r.json())

In [ ]:
# 잘못된 입력 (한 글자) → 422 검증 오류가 정상
bad = test_client.post('/ask', json={'question': 'a'})
print('상태코드:', bad.status_code, '(422면 입력 검증이 정상 작동)')

## Part 4. 과제 — 나만의 학습 도우미 만들기

`build_prompt`의 역할 설명을 바꿔 나만의 AI 도우미를 만드세요.

예시 역할: Python 문법 튜터 / 취업 면접 질문 생성기 / 네트워크 기초 설명 도우미 / 프로젝트 아이디어 추천

제출물: 수정한 프롬프트, 테스트 질문 3개와 응답, 실행 화면 캡처

In [ ]:
def my_prompt(question: str) -> str:
    # TODO: 아래 역할 설명을 자신만의 도우미로 바꿔보세요
    return f'''너는 (여기에 역할을 적으세요. 예: Python 문법 튜터)다.
아래 질문에 친절하고 정확하게 답하라.

질문: {question}'''


def my_assistant(question: str) -> str:
    resp = client.models.generate_content(model=GEMINI_MODEL, contents=my_prompt(question))
    return resp.text


# 테스트 질문 3개를 바꿔가며 실행해 보세요
for q in ['질문 1을 적으세요', '질문 2', '질문 3']:
    print('Q:', q)
    print('A:', my_assistant(q))
    print('-' * 40)